Perfect. This is the **third and final part** of motion blur.

We already updated:

1. **Camera** → creates ray with time
2. **Sphere** → moves using ray time

Now:

3. **Materials** must preserve time after bounce.

Why?

Because after first hit:

Old:

```text
Ray(time=0.42) hits sphere
↓
scattered ray loses time
↓
new ray defaults to time=0
```

Wrong.

That would make the second bounce happen in the wrong world state.

We must preserve:

$$
t_{out}=t_{in}
$$

Same physical instant.

---

# Core Formula

Before:

Scattered ray:

$$
R_{out}(s)=P+sD'
$$

After:

Scattered ray:

$$
R_{out}(s,\tau)=P+sD'
$$

with:

$$
\tau = R_{in}.time()
$$

This keeps time consistent.

---

# Learning Table

| Material         | Formula            | Purpose              |
| ---------------- | ------------------ | -------------------- |
| Lambertian       | $D=N+\omega$       | diffuse bounce       |
| Metal            | $R=V-2(V\cdot N)N$ | mirror bounce        |
| Dielectric       | Snell’s law        | refraction           |
| Time propagation | $t_{out}=t_{in}$   | preserve motion blur |

---

# 1. Lambertian (Updated)

```python
import random

from util.week1.vec3 import random_unit_vector
from util.week1.ray import Ray
from util.week1.material import Material


class Lambertian(Material):

    def __init__(self, albedo):

        # ==========================================
        # Surface color
        # ==========================================
        self.albedo = albedo

    def scatter(self, r_in, rec):

        # ==========================================
        # STEP 1:
        # Diffuse random bounce
        #
        # Formula:
        #
        # D = N + random_unit_vector()
        # ==========================================
        scatter_direction = rec.normal + random_unit_vector()

        # ==========================================
        # STEP 2:
        # Degenerate fix
        #
        # If vector becomes zero
        # ==========================================
        if scatter_direction.near_zero():
            scatter_direction = rec.normal

        # ==========================================
        # STEP 3:
        # Create scattered ray
        #
        # OLD:
        # Ray(rec.p, scatter_direction)
        #
        # NEW:
        # Preserve incoming time
        #
        # t_out = t_in
        # ==========================================
        scattered = Ray(
            rec.p,
            scatter_direction,
            r_in.time()
        )

        # ==========================================
        # STEP 4:
        # Color attenuation
        # ==========================================
        attenuation = self.albedo

        return scattered, attenuation
```

---

# 2. Metal (Updated)

```python
from util.week1.vec3 import (
    unit_vector,
    reflect,
    random_unit_vector
)
from util.week1.ray import Ray
from util.week1.material import Material


class Metal(Material):

    def __init__(self, albedo, fuzz=0.0):

        self.albedo = albedo
        self.fuzz = min(fuzz, 1.0)

    def scatter(self, r_in, rec):

        # ==========================================
        # STEP 1:
        # Normalize incoming ray
        # ==========================================
        unit_dir = unit_vector(
            r_in.direction()
        )

        # ==========================================
        # STEP 2:
        # Reflection formula
        #
        # R = V - 2(V·N)N
        # ==========================================
        reflected = reflect(
            unit_dir,
            rec.normal
        )

        # ==========================================
        # STEP 3:
        # Add fuzz
        # ==========================================
        scattered_dir = (
            reflected
            + self.fuzz * random_unit_vector()
        )

        # ==========================================
        # STEP 4:
        # Preserve time
        #
        # NEW:
        # r_in.time()
        # ==========================================
        scattered = Ray(
            rec.p,
            scattered_dir,
            r_in.time()
        )

        attenuation = self.albedo

        return scattered, attenuation
```

---

# 3. Dielectric (Updated)

```python
import math
import random

from util.week1.vec3 import (
    Vec3,
    unit_vector,
    reflect,
    refract,
    dot
)

from util.week1.ray import Ray


class Dielectric:

    def __init__(self, refraction_index):

        self.refraction_index = refraction_index

    # ==========================================
    # Schlick approximation
    # ==========================================
    def reflectance(self, cosine, ref_idx):

        r0 = (1 - ref_idx) / (1 + ref_idx)
        r0 = r0 * r0

        return r0 + (
            1 - r0
        ) * pow(
            (1 - cosine),
            5
        )

    def scatter(self, r_in, rec):

        # ==========================================
        # STEP 1:
        # Glass absorbs nothing
        # ==========================================
        attenuation = Vec3(
            1.0,
            1.0,
            1.0
        )

        # ==========================================
        # STEP 2:
        # Refraction ratio
        # ==========================================
        if rec.front_face:
            ri = 1.0 / self.refraction_index
        else:
            ri = self.refraction_index

        # ==========================================
        # STEP 3:
        # Normalize direction
        # ==========================================
        unit_dir = unit_vector(
            r_in.direction()
        )

        # ==========================================
        # STEP 4:
        # cos(theta)
        # ==========================================
        cos_theta = min(
            dot(-unit_dir, rec.normal),
            1.0
        )

        sin_theta = math.sqrt(
            1.0 - cos_theta * cos_theta
        )

        # ==========================================
        # STEP 5:
        # Total internal reflection
        # ==========================================
        cannot_refract = (
            ri * sin_theta > 1.0
        )

        # ==========================================
        # STEP 6:
        # Choose reflect or refract
        # ==========================================
        if (
            cannot_refract
            or random.random()
            < self.reflectance(cos_theta, ri)
        ):

            direction = reflect(
                unit_dir,
                rec.normal
            )

        else:

            direction = refract(
                unit_dir,
                rec.normal,
                ri
            )

        # ==========================================
        # STEP 7:
        # Preserve time
        #
        # NEW:
        # r_in.time()
        # ==========================================
        scattered = Ray(
            rec.p,
            direction,
            r_in.time()
        )

        return scattered, attenuation
```

---

# What changed?

| Before              | After                          |
| ------------------- | ------------------------------ |
| `Ray(rec.p, dir)`   | `Ray(rec.p, dir, r_in.time())` |
| time lost           | time preserved                 |
| later bounces wrong | later bounces correct          |

---

# Final motion blur pipeline

Now fully correct:

$$
Camera \to Ray(t)
$$

$$
Sphere \to Center(t)
$$

$$
Hit \to Scatter(t)
$$

This means:

Every bounce stays at the same instant in time.

That is exactly the physically correct motion blur model from the book.


In [17]:
import sys 
sys.path.append("../../../Ray Tracing next Week")

# Lambertian

In [18]:
import random

from util.week1.vec3 import random_unit_vector
from util.week2.ray import Ray
from util.week1.material import Material


class Lambertian(Material):

    def __init__(self, albedo):

        # ==========================================
        # Surface color
        # ==========================================
        self.albedo = albedo

    def scatter(self, r_in, rec):

        # ==========================================
        # STEP 1:
        # Diffuse random bounce
        #
        # Formula:
        #
        # D = N + random_unit_vector()
        # ==========================================
        scatter_direction = rec.normal + random_unit_vector()

        # ==========================================
        # STEP 2:
        # Degenerate fix
        #
        # If vector becomes zero
        # ==========================================
        if scatter_direction.near_zero():
            scatter_direction = rec.normal

        # ==========================================
        # STEP 3:
        # Create scattered ray
        #
        # OLD:
        # Ray(rec.p, scatter_direction)
        #
        # NEW:
        # Preserve incoming time
        #
        # t_out = t_in
        # ==========================================
        scattered = Ray(
            rec.p,
            scatter_direction,
            r_in.time()
        )

        # ==========================================
        # STEP 4:
        # Color attenuation
        # ==========================================
        attenuation = self.albedo

        return scattered, attenuation

# Metal


In [19]:
from util.week1.vec3 import (
    unit_vector,
    reflect,
    random_unit_vector
)
from util.week2.ray import Ray
from util.week1.material import Material


class Metal(Material):

    def __init__(self, albedo, fuzz=0.0):

        self.albedo = albedo
        self.fuzz = min(fuzz, 1.0)

    def scatter(self, r_in, rec):

        # ==========================================
        # STEP 1:
        # Normalize incoming ray
        # ==========================================
        unit_dir = unit_vector(
            r_in.direction()
        )

        # ==========================================
        # STEP 2:
        # Reflection formula
        #
        # R = V - 2(V·N)N
        # ==========================================
        reflected = reflect(
            unit_dir,
            rec.normal
        )

        # ==========================================
        # STEP 3:
        # Add fuzz
        # ==========================================
        scattered_dir = (
            reflected
            + self.fuzz * random_unit_vector()
        )

        # ==========================================
        # STEP 4:
        # Preserve time
        #
        # NEW:
        # r_in.time()
        # ==========================================
        scattered = Ray(
            rec.p,
            scattered_dir,
            r_in.time()
        )

        attenuation = self.albedo

        return scattered, attenuation

# Dielectric

In [20]:

import math
import random

from util.week1.vec3 import (
    Vec3,
    unit_vector,
    reflect,
    refract,
    dot
)

from util.week2.ray import Ray


class Dielectric:

    def __init__(self, refraction_index):

        self.refraction_index = refraction_index

    # ==========================================
    # Schlick approximation
    # ==========================================
    def reflectance(self, cosine, ref_idx):

        r0 = (1 - ref_idx) / (1 + ref_idx)
        r0 = r0 * r0

        return r0 + (
            1 - r0
        ) * pow(
            (1 - cosine),
            5
        )

    def scatter(self, r_in, rec):

        # ==========================================
        # STEP 1:
        # Glass absorbs nothing
        # ==========================================
        attenuation = Vec3(
            1.0,
            1.0,
            1.0
        )

        # ==========================================
        # STEP 2:
        # Refraction ratio
        # ==========================================
        if rec.front_face:
            ri = 1.0 / self.refraction_index
        else:
            ri = self.refraction_index

        # ==========================================
        # STEP 3:
        # Normalize direction
        # ==========================================
        unit_dir = unit_vector(
            r_in.direction()
        )

        # ==========================================
        # STEP 4:
        # cos(theta)
        # ==========================================
        cos_theta = min(
            dot(-unit_dir, rec.normal),
            1.0
        )

        sin_theta = math.sqrt(
            1.0 - cos_theta * cos_theta
        )

        # ==========================================
        # STEP 5:
        # Total internal reflection
        # ==========================================
        cannot_refract = (
            ri * sin_theta > 1.0
        )

        # ==========================================
        # STEP 6:
        # Choose reflect or refract
        # ==========================================
        if (
            cannot_refract
            or random.random()
            < self.reflectance(cos_theta, ri)
        ):

            direction = reflect(
                unit_dir,
                rec.normal
            )

        else:

            direction = refract(
                unit_dir,
                rec.normal,
                ri
            )

        # ==========================================
        # STEP 7:
        # Preserve time
        #
        # NEW:
        # r_in.time()
        # ==========================================
        scattered = Ray(
            rec.p,
            direction,
            r_in.time()
        )

        return scattered, attenuation